#### TECHNICAL GUIDES / DATA SCIENCE & AI

# EXCEL PARA ANALISIS DE DATOS

Carlos A. Ugarte

#### TECHNICAL GUIDES / RESEARCH - EDUCATION - CONSULTING

Cada ejercicio genera sus datos. Las operaciones de interfaz se representan mediante equivalentes Python, no automatización de Excel.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog, brentq
from IPython.display import display
pd.options.display.float_format = "{:.4f}".format
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["axes.formatter.useoffset"] = False

## 1. Limpieza de datos
Herramientas para eliminar registros duplicados, corregir texto, gestionar errores y filtrar registros según criterios.

### Quitar duplicados
Datos > Quitar duplicados. Definir la clave antes de eliminar; conservar el original y revisar conflictos.

In [ ]:
raw=pd.DataFrame({'id':[1,1,2,3], 'cliente':[' Ana  ',' Ana  ','Luis','Eva'], 'ventas':[12000,12000,8000,15000]})
limpio=raw.drop_duplicates().copy()
display(raw,limpio)
assert len(limpio)==3

### Texto en columnas; Buscar y reemplazar
Datos > Texto en columnas: separar por delimitadores. Ctrl+H permite sustituir caracteres o errores. Verificar el número de campos y conservar códigos como texto.

In [ ]:
s=pd.Series(['001,Ana','002,Luis'])
display(s.str.split(',',expand=True).rename(columns={0:'codigo',1:'nombre'}))
display(pd.Series(['$1.200','$800']).str.replace('$','',regex=False).str.replace('.','',regex=False).astype(int))

### TRIM / ESPACIOS y CLEAN / LIMPIAR
ESPACIOS elimina espacios exteriores y repetidos internos; LIMPIAR elimina caracteres ASCII de control. Los espacios Unicode requieren tratamiento adicional.

In [ ]:
s=pd.Series(['  Ana   Pérez  ','Luis\tGómez','Eva\n'])
display(pd.DataFrame({'original':s,'limpiar':s.str.replace(r'[\x00-\x1f]','',regex=True),'espacios':s.str.replace(r' +',' ',regex=True).str.strip(' ')}))

### IFERROR / SI.ERROR; filtros avanzados
SI.ERROR(valor; alternativa) devuelve una alternativa ante errores. No confundir ausencia con cero. Datos > Avanzadas: criterios combinados AND/OR.

In [ ]:
a=pd.Series([100,200,300]); b=pd.Series([2,0,5])
resultado=a.div(b.replace(0,np.nan))
display(pd.DataFrame({'resultado':resultado,'estado':np.where(b.eq(0),'Error: divisor cero','OK')}))
display(limpio.loc[(limpio.ventas>10000)&(limpio.id!=1)])

## 2. Transformación de datos
Funciones de texto y fecha; conexión, combinación y transformación de fuentes con Power Query.

### LEFT, RIGHT, MID; UPPER, LOWER, PROPER
IZQUIERDA, DERECHA y EXTRAE extraen caracteres; Excel cuenta posiciones desde 1, Python desde 0. MAYUSC, MINUSC y NOMPROPIO modifican capitalización.

In [ ]:
s=pd.Series(['AR-001-Ana','BR-002-Luis'])
display(pd.DataFrame({'izquierda':s.str[:2],'derecha':s.str[-3:],'extrae':s.str[3:6],'mayusculas':s.str.upper(),'minusculas':s.str.lower(),'nombre_propio':s.str.title()}))

### NOW, EDATE, WORKDAY
AHORA devuelve fecha y hora actuales; FECHA.MES suma o resta meses; DIA.LAB excluye fines de semana y opcionalmente feriados. Se usa fecha fija para reproducibilidad.

In [ ]:
fecha=pd.Timestamp('2026-01-30')
feriados=pd.to_datetime(['2026-02-02'])
display(pd.DataFrame({'ahora':[pd.Timestamp.now().floor('s')],'fecha_base':[fecha],'mas_un_mes':[fecha+pd.DateOffset(months=1)],'menos_un_mes':[fecha-pd.DateOffset(months=1)],'dos_dias_habiles':[fecha+pd.offsets.BusinessDay(2)],'con_feriados':[fecha+pd.offsets.CustomBusinessDay(2,holidays=feriados)]}))

### Power Query: conectar, combinar y anular dinamización
Datos > Obtener datos: importar CSV o bases. Combinar consultas equivale a join; anexar equivale a concatenar filas; anular dinamización convierte columnas en filas. Validar cardinalidad.

In [ ]:
from io import StringIO
ventas=pd.read_csv(StringIO('id,mes,importe\n1,Ene,12000\n2,Ene,8000\n1,Feb,15000'))
clientes=pd.DataFrame({'id':[1,2],'region':['Norte','Sur']})
extra=pd.DataFrame({'id':[2],'mes':['Feb'],'importe':[9000]})
ventas=pd.concat([ventas,extra],ignore_index=True)
modelo=ventas.merge(clientes,on='id',validate='many_to_one')
ancho=ventas.pivot(index='id',columns='mes',values='importe').reset_index()
display(modelo,ancho,ancho.melt(id_vars='id',var_name='mes',value_name='importe'))

## 3. Funciones de análisis de datos
Estadística descriptiva, lógica y búsqueda de referencias.

### AVERAGE, MEDIAN, MODE; VAR.S, VAR.P; STDEV.S, STDEV.P
PROMEDIO, MEDIANA, MODA.UNO; VAR.S y VAR.P; DESVEST.M y DESVEST.P. La varianza muestral usa n−1 y la poblacional n. MODA.UNO devuelve una moda; revisar empates.

In [ ]:
x=pd.Series([10,12,12,15,21])
display(pd.Series({'media':x.mean(),'mediana':x.median(),'moda':x.mode().iloc[0],'var_muestra':x.var(ddof=1),'var_poblacion':x.var(ddof=0),'desv_muestra':x.std(ddof=1),'desv_poblacion':x.std(ddof=0)}).to_frame('valor'))

### IF, AND, OR e IF anidado
SI, Y, O. Las condiciones múltiples se evalúan con operadores booleanos; los umbrales deben estar documentados.

In [ ]:
v=pd.Series([5000,10000,15000]); mora=pd.Series([0,30,10])
display(pd.DataFrame({'ventas':v,'mora':mora,'SI':np.where(v>10000,'Alta','Normal'),'Y':(v>10000)&(mora<15),'O':(v>10000)|(mora>20),'SI_anidado':np.select([v>10000,v>=8000],['Alta','Media'],default='Baja')}))

### VLOOKUP, HLOOKUP, INDEX + MATCH, XLOOKUP
BUSCARV y BUSCARH buscan vertical u horizontalmente; INDICE + COINCIDIR permite buscar por posición; BUSCARX admite retorno flexible y valor alternativo. Usar coincidencia exacta (FALSO o 0) y claves únicas.

In [ ]:
catalogo=pd.DataFrame({'codigo':['A','B','C'],'precio':[12.5,20,35]})
consulta=pd.Series(['B','D','A'])
vertical=consulta.map(catalogo.set_index('codigo').precio)
horizontal=catalogo.set_index('codigo').T
pos=catalogo.codigo.tolist().index('B')
display(pd.DataFrame({'codigo':consulta,'BUSCARV':vertical,'BUSCARX':vertical.astype(object).where(vertical.notna(),'No encontrado')}))
print(f'BUSCARH: {horizontal.loc["precio","B"]:.4f}; INDICE+COINCIDIR: {catalogo.precio.iloc[pos]:.4f}')

## 4. Visualización de datos
Insertar > Gráficos recomendados; elegir según objetivo: comparación, tendencia, relación o composición.

### Columnas, barras, líneas, dispersión y circular
Barras comparan categorías; líneas muestran evolución; dispersión analiza relaciones; circular muestra partes de un total no negativo. Evitar interpretar correlación como causalidad.

In [ ]:
datos=pd.DataFrame({'mes':['Ene','Feb','Mar','Abr'],'ventas':[12000,15000,13000,18000],'publicidad':[1000,1600,1200,1900]})
for tipo in ['bar','barh','line','scatter','pie']:
    fig,ax=plt.subplots(figsize=(5,3))
    if tipo=='scatter': ax.scatter(datos.publicidad,datos.ventas); ax.set_xlabel('Publicidad'); ax.set_ylabel('Ventas')
    elif tipo=='pie': ax.pie(datos.ventas,labels=datos.mes,autopct='%.4f%%')
    else: datos.set_index('mes').ventas.plot(kind=tipo,ax=ax)
    ax.set_title(tipo); fig.tight_layout(); display(fig); plt.close(fig)

### Formato condicional y minigráficos
Inicio > Formato condicional: barras de datos, escalas de color, conjuntos de iconos y regla mayor que 10.000. Insertar > Minigráficos: tendencias dentro de celdas.

In [ ]:
tabla=datos[['mes','ventas']].copy(); tabla['icono']=np.select([tabla.ventas>=15000,tabla.ventas>=10000],['Alto','Medio'],default='Bajo')
display(tabla.style.format(precision=4).bar(subset=['ventas']).map(lambda x:'font-weight:bold' if x>10000 else '',subset=['ventas']))
display(tabla.style.background_gradient(subset=['ventas']).format(precision=4))
fig,ax=plt.subplots(figsize=(3,0.8)); ax.plot(datos.ventas); ax.axis('off'); display(fig); plt.close(fig)

## 5. Herramientas avanzadas de análisis
Tablas dinámicas, análisis de hipótesis, Solver y modelos relacionales con Power Pivot.

### Tablas dinámicas, segmentadores y campos calculados
Insertar > Tabla dinámica: resumir por dimensiones. Segmentadores filtran visualmente. Un campo calculado puede expresar margen; para porcentajes agregados usar razón de sumas.

In [ ]:
m=modelo.assign(costo=modelo.importe*.65)
m['margen']=m.importe-m.costo
display(m.pivot_table(index='region',columns='mes',values='importe',aggfunc='sum',margins=True))
display(m.loc[m.region.eq('Norte')].groupby('mes').importe.sum().to_frame())
g=m.groupby('region')[['importe','margen']].sum(); g['margen_pct']=100*g.margen/g.importe; display(g)

### Buscar objetivo, Administrador de escenarios y Tablas de datos
Datos > Análisis de hipótesis. Buscar objetivo cambia una entrada para alcanzar una meta; escenarios comparan supuestos; tablas de datos evalúan una o dos variables.

In [ ]:
def beneficio(q,p=50,c=30,f=1000): return q*(p-c)-f
q=brentq(lambda q:beneficio(q)-2000,0,1000)
print(f'Unidades para beneficio 2000: {q:.4f}')
esc=pd.DataFrame({'escenario':['Pesimista','Base','Optimista'],'cantidad':[80,150,220],'precio':[45,50,55]}); esc['beneficio']=beneficio(esc.cantidad,esc.precio); display(esc)
display(pd.DataFrame([[beneficio(q,p) for p in [45,50,55]] for q in [80,150,220]],index=[80,150,220],columns=[45,50,55]))

### Solver
Optimizar una función objetivo bajo restricciones. Ejemplo: minimizar costo de compra con demanda mínima de 100 unidades y capacidad máxima de 60 del proveedor barato. Solver puede requerir activar el complemento.

In [ ]:
res=linprog([4,6],A_ub=[[-1,-1]],b_ub=[-100],bounds=[(0,60),(0,None)],method='highs')
assert res.success and np.isclose(res.fun,480)
print(f'Proveedor A: {res.x[0]:.4f}; B: {res.x[1]:.4f}; costo: {res.fun:.4f}')

### Power Pivot y DAX: SUMX, CALCULATE
Power Pivot crea relaciones entre tablas y medidas DAX. SUMX(Ventas; Ventas[Cantidad]*Ventas[Precio]) itera filas; CALCULATE modifica el contexto de filtro. Python reproduce el cálculo, no el motor DAX.

In [ ]:
dim=pd.DataFrame({'id':[1,2],'region':['Norte','Sur']})
fact=pd.DataFrame({'id':[1,2,1],'cantidad':[10,20,15],'precio':[100,80,120]})
f=fact.merge(dim,on='id',validate='many_to_one'); f['importe']=f.cantidad*f.precio
print(f'SUMX total: {f.importe.sum():.4f}; CALCULATE Norte: {f.loc[f.region.eq("Norte"),"importe"].sum():.4f}')

## 6. Atajos de teclado
Atajos del PDF para Excel de escritorio en Windows. Las secuencias de cinta pueden variar según idioma y versión.

- Seleccionar columna: Ctrl + Espacio
- Seleccionar fila: Mayús + Espacio
- Crear tabla: Ctrl + T
- Autosuma: Alt + =
- Insertar hoja: Mayús + F11
- Abrir filtro: Alt + Flecha abajo
- Inmovilizar paneles: Alt + W + F + F (interfaz inglesa)

In [ ]:
display(pd.DataFrame([('Seleccionar columna', 'Ctrl + Espacio'), ('Seleccionar fila', 'Mayús + Espacio'), ('Crear tabla', 'Ctrl + T'), ('Autosuma', 'Alt + ='), ('Insertar hoja', 'Mayús + F11'), ('Abrir filtro', 'Alt + Flecha abajo'), ('Inmovilizar paneles', 'Alt + W + F + F (interfaz inglesa)')],columns=["Acción","Atajo"]))
# Equivalentes de selección, suma y filtro; no emulan pulsaciones.
display(ventas["importe"],ventas.iloc[0],ventas.loc[ventas.importe>10000]); print(ventas.importe.sum())

## 7. Consejos profesionales para un análisis eficiente
Usar nombres definidos (Fórmulas > Definir nombre), validación de datos (listas desplegables), fórmulas combinadas, copias de respaldo y hojas organizadas con colores, comentarios y diseño limpio.

### Rangos nombrados, validación y fórmulas combinadas
Ejemplo original: =IF(AND(A1>10,B1<5),"Yes","No"); en español =SI(Y(A1>10;B1<5);"Sí";"No"). En Python usar variables con nombres descriptivos y dominios permitidos.

In [ ]:
rango_ventas=ventas.importe.copy(); estados=pd.Series(['Activo','Inactivo','Erróneo']); permitidos=['Activo','Inactivo']
display(pd.DataFrame({'estado':estados,'valido':estados.isin(permitidos)}))
a=pd.Series([11,8,20]); b=pd.Series([3,2,7]); display(pd.DataFrame({'A':a,'B':b,'resultado':np.where((a>10)&(b<5),'Sí','No')}))

### Respaldo y organización
Trabajar sobre una copia, no sobrescribir datos originales; separar datos brutos, transformación y resultados. Documentar criterios y comentarios. El color de pestañas es una operación visual de Excel.

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path
with TemporaryDirectory() as tmp:
    base=Path(tmp); raw.to_csv(base/'original.csv',index=False); limpio.to_csv(base/'limpio.csv',index=False)
    assert len(pd.read_csv(base/'original.csv'))==4
    display(pd.DataFrame({'archivo':['original.csv','limpio.csv'],'filas':[len(raw),len(limpio)]}))

## Resumen: Excel para el análisis de datos
Eliminar duplicados, espacios y errores; transformar texto y fechas y usar Power Query; aplicar media, mediana y varianza; dominar gráficos, formato condicional y minigráficos; profundizar con tablas dinámicas, Solver y análisis de hipótesis; memorizar atajos; aprovechar Power Query y Power Pivot.

## Wrap Up
Un flujo analítico confiable conecta fuentes, conserva datos brutos, limpia y valida, transforma, calcula indicadores, visualiza y optimiza decisiones. Excel facilita la exploración interactiva; Python aporta automatización, pruebas y reproducibilidad. No son sustitutos exactos: DAX depende del contexto de filtro y Power Query registra pasos de transformación.

## Takeaways
1. Definir claves antes de deduplicar. 2. No ocultar errores con ceros. 3. Validar cardinalidad de uniones. 4. Distinguir muestra y población. 5. Usar búsquedas exactas. 6. Elegir gráficos según la pregunta. 7. Calcular razones sobre agregados. 8. Documentar restricciones de optimización. 9. Conservar datos originales. 10. Automatizar con pruebas reproducibles.

## Glosario

**ETL**: Extracción, transformación y carga.

**Power Query**: Motor de importación y transformación.

**Power Pivot**: Modelo tabular y relaciones.

**DAX**: Lenguaje de expresiones analíticas.

**Cardinalidad**: Relación entre cantidades de claves.

**Join**: Unión por claves.

**Append**: Anexión de filas.

**Unpivot**: Conversión de columnas en filas.

**Varianza**: Dispersión cuadrática.

**Desviación estándar**: Raíz de la varianza.

**Moda**: Valor más frecuente.

**Segmentador**: Filtro visual.

**Tabla dinámica**: Resumen multidimensional.

**Solver**: Optimizador sujeto a restricciones.

**Buscar objetivo**: Búsqueda de entrada para una meta.

**Escenario**: Conjunto de supuestos.

**Minigráfico**: Gráfico compacto en una celda.

**Validación**: Control de entradas permitidas.

**Contexto de filtro**: Subconjunto que afecta una medida DAX.

**Rango nombrado**: Referencia con nombre descriptivo.